In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

dataset_1 = pd.read_csv('collegeISLPDataAnalysis2027_numeric.csv')
dataset_2 = pd.read_csv('digitsDataAnalysis2027.csv')

# Resuable split
def split_dataset(D, ratio=(0.6, 0.2, 0.2), random_state=None):
    
    X, y = D
    X = np.asarray(X)
    y = np.asarray(y)

    rng = np.random.default_rng(random_state)
    indices = rng.permutation(len(X))

    n_train = int(ratio[0] * len(X))
    n_validation = int(ratio[1] * len(X))

    train_indices = indices[:n_train]
    validation_indices = indices[n_train:n_train + n_validation]
    test_indices = indices[n_train + n_validation:]

    train = (X[train_indices], y[train_indices])
    validation = (X[validation_indices], y[validation_indices])
    test = (X[test_indices], y[test_indices])

    return train, validation, test

In [ ]:
# Linear Discriminant Analysis (College Dataset)

data = dataset_1
X = data.drop(columns=["Grad.Rate"]).to_numpy(dtype=float)
y = data["Grad.Rate"].to_numpy(dtype=int)

train, validation, test = split_dataset((X, y), [.6, .2, .2], random_state=42)
X_train, y_train = train
X_val, y_val = validation
X_test, y_test = test

# Standardise using training-set statistics only.
mean = X_train.mean(axis=0)
scale = X_train.std(axis=0)
scale[scale == 0] = 1
X_train = (X_train - mean) / scale
X_val = (X_val - mean) / scale
X_test = (X_test - mean) / scale

class LDA:
    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        self.classes = np.unique(y)
        self.means = np.array([X[y == label].mean(axis=0) for label in self.classes])
        self.priors = np.array([np.mean(y == label) for label in self.classes])

        centered = np.concatenate([X[y == label] - self.means[i]
                                    for i, label in enumerate(self.classes)])
        covariance = centered.T @ centered / (len(X) - len(self.classes))
        self.covariance_inverse = np.linalg.pinv(covariance)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        predictions = []
        for row in X:
            differences = self.means - row
            scores = (-0.5 * np.sum((differences @ self.covariance_inverse) * differences, axis=1)
                      + np.log(self.priors))
            predictions.append(self.classes[np.argmax(scores)])
        return np.asarray(predictions)

lda = LDA().fit(X_train, y_train)
y_val_pred = lda.predict(X_val)
y_test_pred = lda.predict(X_test)
validation_accuracy = np.mean(y_val_pred == y_val)
test_accuracy = np.mean(y_test_pred == y_test)
test_mse = np.mean((y_test - y_test_pred) ** 2)

print(f"Validation accuracy: {validation_accuracy:.3f}")
print(f"Test accuracy: {test_accuracy:.3f}")
print(f"Test MSE: {test_mse:.3f}")

classes = np.unique(y)
confusion = np.zeros((len(classes), len(classes)), dtype=int)
for actual, predicted in zip(y_test, y_test_pred):
    confusion[np.where(classes == actual)[0][0],
              np.where(classes == predicted)[0][0]] += 1

plt.figure(figsize=(6, 5))
plt.imshow(confusion, cmap="Blues")
plt.xlabel("Predicted Grad.Rate class")
plt.ylabel("Actual Grad.Rate class")
plt.title("LDA confusion matrix - College dataset")
plt.xticks(range(len(classes)), classes)
plt.yticks(range(len(classes)), classes)
for row in range(len(classes)):
    for column in range(len(classes)):
        plt.text(column, row, confusion[row, column], ha="center", va="center")
plt.colorbar(label="Number of colleges")
plt.tight_layout()
plt.show()

In [ ]:
# Linear Discriminant Analysis (Digit Dataset) 

data = dataset_2

X = data.drop(columns=["digit"]).to_numpy(dtype=float)
y = data["digit"].to_numpy()

train, validation, test = split_dataset((X, y), [.6, .2, .2], random_state=42)
X_train, y_train = train
X_val, y_val = validation
X_test, y_test = test

mean = X_train.mean(axis=0)
scale = X_train.std(axis=0)
scale[scale == 0] = 1
X_train = (X_train - mean) / scale
X_val = (X_val - mean) / scale
X_test = (X_test - mean) / scale

class LDA:

    def __init__(self, n_components):
        self.n_components = n_components
        self.linear_discriminants = None
        self.mean_overall = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)

        class_labels = np.unique(y)
        self.classes = class_labels
        n_features = X.shape[1]

        self.mean_overall = np.mean(X, axis=0)
        self.means = np.array([X[y == label].mean(axis=0) for label in class_labels])
        self.priors = np.array([np.mean(y == label) for label in class_labels])

        S_W = np.zeros((n_features, n_features), dtype=float)
        S_B = np.zeros((n_features, n_features), dtype=float)

        for class_label in class_labels:
            X_class = X[y == class_label]
            mean_class = np.mean(X_class, axis=0)

            # Within-class scatter
            centered_class = X_class - mean_class
            S_W += centered_class.T @ centered_class

            # Between-class scatter
            mean_difference = (mean_class - self.mean_overall).reshape(-1, 1)
            S_B += X_class.shape[0] * (mean_difference @ mean_difference.T)

        # Inverse
        self._covariance = S_W / (len(X) - len(class_labels))
        S_W_inverse = np.linalg.pinv(S_W)
        A = S_W_inverse @ S_B

        eigenvalues, eigenvectors = np.linalg.eig(A)

        # Eigenvalues & Eigenvectors
        eigenvalues = np.real(eigenvalues)
        eigenvectors = np.real(eigenvectors)

        indices = np.argsort(eigenvalues)[::-1]
        eigenvectors = eigenvectors[:, indices]

        # Each column is one discriminant direction.
        self.linear_discriminants = eigenvectors[:, :self.n_components]

        return self

    def transform(self, X):

        X = np.asarray(X, dtype=float)
        return (X - self.mean_overall) @ self.linear_discriminants

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        covariance = np.linalg.pinv(self._covariance)
        predictions = []
        for row in X:
            differences = self.means - row
            scores = (-0.5 * np.sum((differences @ covariance) * differences, axis=1)
                      + np.log(self.priors))
            predictions.append(self.classes[np.argmax(scores)])
        return np.asarray(predictions)


lda = LDA(n_components=2)
lda.fit(X_train, y_train)

y_val_pred = lda.predict(X_val)
y_test_pred = lda.predict(X_test)
validation_accuracy = np.mean(y_val_pred == y_val)
test_accuracy = np.mean(y_test_pred == y_test)
test_mse = np.mean((y_test - y_test_pred) ** 2)

print(f"Validation accuracy: {validation_accuracy:.3f}")
print(f"Test accuracy: {test_accuracy:.3f}")
print(f"Test MSE: {test_mse:.3f}")

X_projected = lda.transform((X - mean) / scale)

print("Shape of X:", X.shape)
print("Shape of transformed X:", X_projected.shape)



plt.figure(figsize=(8, 6))
plt.scatter(
    X_projected[:, 0],
    X_projected[:, 1],
    c=y,
    edgecolor="none",
    alpha=0.8,
    cmap="tab10"
)
plt.xlabel("Linear Discriminant 1")
plt.ylabel("Linear Discriminant 2")
plt.colorbar(label="Digit")
plt.title("LDA projection of the digit dataset")
plt.show()
plt.savefig('LDA Digit Dataset')

In [ ]:
# KNN (College Dataset)

file = 'collegeISLPDataAnalysis2027_numeric.csv'
data = pd.read_csv(file, sep=',', header=0)
print(f'attributes: {data.columns}')
print(f'first object: {data.values[0]}')
print(f'size of the data set: |D| = {len(data.values)}')

# GradRate
X = data.drop(columns=['Grad.Rate']).to_numpy(dtype=float)
y = data['Grad.Rate'].to_numpy(dtype=int)
print(f'X.shape = {X.shape}, y.shape = {y.shape}')

train, validation, test = split_dataset((X, y), [.6, .2, .2], random_state=42)
X_train, y_train = train
X_val, y_val = validation
X_test, y_test = test
print(f"|D_train| = {len(X_train)}, |D_val|={len(X_val)}, |D_test| = {len(X_test)}")

# Standardise using training-set statistics only
mean = X_train.mean(axis=0)
scale = X_train.std(axis=0)
scale[scale == 0] = 1
X_train = (X_train - mean) / scale
X_val = (X_val - mean) / scale
X_test = (X_test - mean) / scale

class KNNClassifier:
    def __init__(self, k=5):
        if k < 1:
            raise ValueError('k must be at least 1')
        self.k = k
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        if X.ndim != 2 or len(X) != len(y):
            raise ValueError('X must be 2-D and have the same number of rows as y')
        if self.k > len(X):
            raise ValueError('k cannot be larger than the training set')
        self.X_train = X
        self.y_train = y
        return self

    def predict(self, X):
        if self.X_train is None or self.y_train is None:
            raise RuntimeError('Call fit before predict')
        X = np.asarray(X, dtype=float)
        predictions = []
        for row in X:
            distances = np.sqrt(np.sum((self.X_train - row) ** 2, axis=1))
            nearest = np.argsort(distances)[:self.k]
            labels, counts = np.unique(self.y_train[nearest], return_counts=True)
            predictions.append(labels[np.argmax(counts)])
        return np.asarray(predictions)

def mse(y_true, y_pred):
    return np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)

k_values = [1, 3, 5, 7, 11, 15, 21]
validation_mse = []
for k in k_values:
    model = KNNClassifier(k=k).fit(X_train, y_train)
    validation_mse.append(mse(y_val, model.predict(X_val)))

best_k = k_values[int(np.argmin(validation_mse))]
knn = KNNClassifier(k=best_k).fit(X_train, y_train)
y_test_pred = knn.predict(X_test)
test_accuracy = np.mean(y_test_pred == y_test)
test_mse = mse(y_test, y_test_pred)

print(f'Validation MSE by k: {dict(zip(k_values, validation_mse))}')
print(f'Best k: {best_k}')
print(f'Test accuracy: {test_accuracy:.3f}')
print(f'Test MSE: {test_mse:.3f}')

plt.figure(figsize=(8, 5))
plt.plot(k_values, validation_mse, marker='o', linewidth=2)
plt.axvline(best_k, color='red', linestyle='--', label=f'Best k = {best_k}')
plt.scatter(best_k, min(validation_mse), color='red', zorder=3)
plt.xlabel('Number of neighbours (k)')
plt.ylabel('Validation MSE')
plt.title('KNN validation MSE for different values of k')
plt.xticks(k_values)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Compare the predicted and actual graduation-rate classes.
classes = np.unique(y)
confusion = np.zeros((len(classes), len(classes)), dtype=int)
for actual, predicted in zip(y_test, y_test_pred):
    actual_index = np.where(classes == actual)[0][0]
    predicted_index = np.where(classes == predicted)[0][0]
    confusion[actual_index, predicted_index] += 1

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(
    range(len(y_test)), y_test,
    label='Actual Grad.Rate', alpha=0.7
)
axes[0].scatter(
    range(len(y_test)), y_test_pred,
    label='Predicted Grad.Rate', alpha=0.7, marker='x'
)
axes[0].set_xlabel('Test observation')
axes[0].set_ylabel('Grad.Rate class')
axes[0].set_title('Actual versus predicted graduation-rate class')
axes[0].set_yticks(classes)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

image = axes[1].imshow(confusion, cmap='Blues')
axes[1].set_xlabel('Predicted Grad.Rate class')
axes[1].set_ylabel('Actual Grad.Rate class')
axes[1].set_title('KNN confusion matrix')
axes[1].set_xticks(range(len(classes)), classes)
axes[1].set_yticks(range(len(classes)), classes)
for row in range(len(classes)):
    for column in range(len(classes)):
        axes[1].text(column, row, confusion[row, column],
                     ha='center', va='center')
fig.colorbar(image, ax=axes[1], label='Number of colleges')
fig.tight_layout()
plt.show()
plt.savefig('KNN College Dataset.png')

In [ ]:
# KNN (Digit Dataset)

data = dataset_2
X = data.drop(columns=['digit']).to_numpy(dtype=float)
y = data['digit'].to_numpy(dtype=int)
print(f'X.shape = {X.shape}, y.shape = {y.shape}')

train, validation, test = split_dataset((X, y), [.6, .2, .2], random_state=42)
X_train, y_train = train
X_val, y_val = validation
X_test, y_test = test
print(f"|D_train| = {len(X_train)}, |D_val| = {len(X_val)}, |D_test| = {len(X_test)}")

# Scale pixel values using training-set statistics only.
mean = X_train.mean(axis=0)
scale = X_train.std(axis=0)
scale[scale == 0] = 1
X_train = (X_train - mean) / scale
X_val = (X_val - mean) / scale
X_test = (X_test - mean) / scale

k_values = [1, 3, 5, 7, 9]
validation_accuracy = []
for k in k_values:
    model = KNNClassifier(k=k).fit(X_train, y_train)
    validation_accuracy.append(np.mean(model.predict(X_val) == y_val))

best_k = k_values[int(np.argmax(validation_accuracy))]
knn_digits = KNNClassifier(k=best_k).fit(X_train, y_train)
y_test_pred = knn_digits.predict(X_test)
test_accuracy = np.mean(y_test_pred == y_test)
test_mse = mse(y_test, y_test_pred)

print(f'Validation accuracy by k: {dict(zip(k_values, validation_accuracy))}')
print(f'Best k: {best_k}')
print(f'Test accuracy: {test_accuracy:.3f}')
print(f'Test MSE: {test_mse:.3f}')

plt.figure(figsize=(8, 5))
plt.plot(k_values, validation_accuracy, marker='o', linewidth=2)
plt.axvline(best_k, color='red', linestyle='--', label=f'Best k = {best_k}')
plt.scatter(best_k, max(validation_accuracy), color='red', zorder=3)
plt.xlabel('Number of neighbours (k)')
plt.ylabel('Validation accuracy')
plt.title('KNN digit validation accuracy for different values of k')
plt.xticks(k_values)
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

classes = np.unique(y)
confusion = np.zeros((len(classes), len(classes)), dtype=int)
for actual, predicted in zip(y_test, y_test_pred):
    actual_index = np.where(classes == actual)[0][0]
    predicted_index = np.where(classes == predicted)[0][0]
    confusion[actual_index, predicted_index] += 1

plt.figure(figsize=(8, 7))
image = plt.imshow(confusion, cmap='Blues')
plt.xlabel('Predicted digit')
plt.ylabel('Actual digit')
plt.title('KNN confusion matrix - Digit dataset')
plt.xticks(range(len(classes)), classes)
plt.yticks(range(len(classes)), classes)
for row in range(len(classes)):
    for column in range(len(classes)):
        plt.text(column, row, confusion[row, column], ha='center', va='center')
plt.colorbar(image, label='Number of images')
plt.tight_layout()
plt.show()


In [ ]:
# Naive Bayes (College Dataset)

data = dataset_1

X = data.drop(columns=['Apps', 'Accept', 'Room.Board'])
y = data['Grad.Rate']

train, _, test = split_dataset((X.to_numpy(), y.to_numpy()), ratio=(0.8, 0, 0.2), random_state=42)
X_train = pd.DataFrame(train[0], columns=X.columns)
y_train = pd.Series(train[1])
X_test = pd.DataFrame(test[0], columns=X.columns)
y_test = pd.Series(test[1])

class NaiveBayes:
    def fit(self, X_train, y_train):
        self.classes = np.unique(y_train)
        self.priors = [len(y_train[y_train == c]) / len(y_train) for c in self.classes]

        self.means = [X_train[y_train == c].mean() for c in self.classes]
        self.stds = [X_train[y_train == c].std().replace(0, 1e-9) for c in self.classes]

    def compute_likelihood(self, row, class_idx):
        likelihood = 1
        for feature in row.index:
            mean = self.means[class_idx][feature]
            std = self.stds[class_idx][feature]
            likelihood *= (1 / (np.sqrt(2 * np.pi) * std)) * np.exp((-(row[feature] - mean)**2) / (2 * std**2))

        return likelihood

    def predict(self, X):
        y_pred = []
        for _, row in X.iterrows():
            posteriors = []
            for i in range(len(self.classes)):
                likelihood = self.compute_likelihood(row, i)
                posteriors.append(likelihood * self.priors[i])

            y_pred.append(self.classes[np.argmax(posteriors)])

        return np.array(y_pred)

nb = NaiveBayes()
nb.fit(X_train, y_train)
predictions = nb.predict(X_test)

accuracy = np.mean(predictions == y_test) * 100
mse = np.mean((y_test.to_numpy() - predictions) ** 2)
print(f"Accuracy: {accuracy:.2f}%")
print(f"MSE: {mse:.3f}")

In [ ]:
# Naive Bayes (Digit Dataset)

data = dataset_2

X = data.drop(columns=['digit'])
y = data['digit']

train, _, test = split_dataset((X.to_numpy(), y.to_numpy()), ratio=(0.8, 0, 0.2), random_state=42)
X_train = pd.DataFrame(train[0], columns=X.columns)
y_train = pd.Series(train[1])
X_test = pd.DataFrame(test[0], columns=X.columns)
y_test = pd.Series(test[1])

nb = NaiveBayes()
nb.fit(X_train, y_train)
predictions = nb.predict(X_test)

accuracy = np.mean(predictions == y_test) * 100
mse = np.mean((y_test.to_numpy() - predictions) ** 2)
print(f"Accuracy: {accuracy:.2f}%")
print(f"MSE: {mse:.3f}")


In [ ]:
# Multinomial Logistic Regression

data = dataset_1
X = data.drop(columns=['Apps', 'Accept', 'Room.Board'])
y = data['Grad.Rate']

train, _, test = split_dataset((X.to_numpy(dtype=float), y.to_numpy()),
                                ratio=(0.8, 0, 0.2), random_state=42)
X_train = pd.DataFrame(train[0], columns=X.columns)
y_train = pd.Series(train[1])
X_test = pd.DataFrame(test[0], columns=X.columns)
y_test = pd.Series(test[1])

# Standardise using training-set statistics only.
mean = X_train.mean()
scale = X_train.std().replace(0, 1)
X_train_scaled = (X_train - mean) / scale
X_test_scaled = (X_test - mean) / scale

class MultinomialLogisticRegression:
    def __init__(self, learning_rate=0.1, n_iterations=3000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        self.classes = np.unique(y)
        class_indices = np.searchsorted(self.classes, y)
        y_one_hot = np.eye(len(self.classes))[class_indices]
        X_with_intercept = np.column_stack([np.ones(len(X)), X])
        self.weights = np.zeros((X_with_intercept.shape[1], len(self.classes)))

        for _ in range(self.n_iterations):
            probabilities = self._softmax(X_with_intercept @ self.weights)
            gradient = X_with_intercept.T @ (probabilities - y_one_hot) / len(X)
            self.weights -= self.learning_rate * gradient
        return self

    @staticmethod
    def _softmax(scores):
        scores = scores - np.max(scores, axis=1, keepdims=True)
        exponentials = np.exp(scores)
        return exponentials / exponentials.sum(axis=1, keepdims=True)

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        X_with_intercept = np.column_stack([np.ones(len(X)), X])
        probabilities = self._softmax(X_with_intercept @ self.weights)
        return self.classes[np.argmax(probabilities, axis=1)]

model = MultinomialLogisticRegression()
model.fit(X_train_scaled, y_train)
predictions = model.predict(X_test_scaled)

accuracy = np.mean(predictions == y_test.to_numpy()) * 100
mse = np.mean((y_test.to_numpy() - predictions) ** 2)
print(f"Accuracy: {accuracy:.2f}%")
print(f"Test-error rate: {100 - accuracy:.2f}%")
print(f"MSE: {mse:.3f}")


#### Results & Comparison

Linear Discriminant Analysis (College Dataset):

Shape of X:
Shape of transformed X: 

Linear Discriminant Analysis (Digit Dataset):

Shape of X: (1797, 64)
Shape of transformed X: (1797, 2)

Comparison:

- For the college dataset, LDA yielded a test accuracy of 50%, with a MSE of 0.994.

-----------------------------------------------

KNN (College Dataset):

Best K: 21
Test Accuracy: 0.455
Test MSE: 0.923

KNN (Digit Dataset):

Best K: 1
Test Accuracy: 0.953
Test MSE: 1.233

Comparison:

- The KNN model was vastly more accurate for the Digit dataset, achieving a test accuracy of 0.953 (95.3%), compared to the College model achieving 0.455 (45.5%)

- The Digit dataset had a larger sample size, but more distinct patterns, making KNN classification more effective.

- The College model has a larger neighbourhood of 21, compared to the Digit model having 1 neighbour. From this we can assume that the first image in the Digit dataset was highly informative.

-----------------------------------------------

Naive Bayes (College Dataset):

Accuracy: 100.00%
Test-Error Rate: 0%
MSE: 0.000

Naive Bayes (Digit Dataset):

Accuracy: 63.33%
Test-Error Rate: 33.67%
MSE: 6.775


Comparison:

- The Naive Bayes model was superior on the College dataset, achieving 100.00% overall accuracy, compared to the Digit dataset which achieved 63.33% accuracy.

- The College model may be overfitting or has a favourable split, despite the low variance of Naive Bayes. The Digit dataset appears less likely to be overfitting due to its lower performance.

- The higher test-error rate shows that Naive Bayes struggles more with digit images, rather than concrete numerical values.
-----------------------------------------------

Multinomial Logistic Regression (College Dataset):

Accuracy: 47.1%
Test-Accuracy: 50.00%
MSE: 0.994


Multinomial Logistic Regression (Digit Dataset):

Accuracy: 94.4%
Test-Accuracy: 93.9%
MSE: 1.717

- LDA performs considerably better on the Digit dataset, with approximately 93.9% test accuracy.
The College dataset achieves only 50.0% test accuracy, indicating weaker class separation.

- The Digit model generalises well because validation and test accuracy are very similar: 94.4% vs 93.9%.

- The College model performs slightly better on the test set than validation set, but both results remain relatively low.
- he Digit dataset has a higher MSE because its target labels range from 0 to 9, so misclassifications can produce larger squared errors. MSE should therefore not be compared directly across these datasets; accuracy is the more meaningful classification metric.